In [1]:
import os
import sys

current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)

from src.models.utils import *
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go



In [2]:
plants_temperature_path = os.path.join(project_root, "data", "raw", "PlantsTemperature_View_original.csv")
p_temp_df = pd.read_csv(plants_temperature_path, encoding='utf-8')

p_temp_df["HourNo"] = p_temp_df["HourNo"].astype(int)
p_temp_df["Date"] = p_temp_df["Date"].apply(jalali_to_gregorian_fast)
p_temp_df["datetime"] = pd.to_datetime(p_temp_df["Date"]) + pd.to_timedelta(p_temp_df["HourNo"], unit='h')

is_env = p_temp_df["Code"] == "SCADAF"
tempsens_df = p_temp_df[~is_env]

In [3]:
csv_semi_processed_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
semi_integrated_df = pd.read_csv(csv_semi_processed_path, encoding='utf-8')
semi_integrated_df['date'] = pd.to_datetime(semi_integrated_df['date'])
semi_integrated_df['datetime'] = semi_integrated_df['date'] + pd.to_timedelta(semi_integrated_df['hour'], unit='h')
semi_integrated_df = semi_integrated_df.rename(columns={"name": "PowerPlantName"})

# semi_integrated_df = semi_integrated_df[(semi_integrated_df["is_good_peak"] >= 3)]


In [4]:
df_merged = pd.merge(semi_integrated_df, tempsens_df, on=["datetime", "PowerPlantName"], how="inner")
df_merged = df_merged[['PowerPlantName','code', 'date', 'hour', 'temperature', 'Value', 'generation', 'is_good_peak']].dropna()
df_merged = df_merged.rename(columns={"Value": "sen_temperature"})
df_merged = df_merged.rename(columns={"PowerPlantName": "name"})

In [5]:
joined_df_path = os.path.join(project_root, "data", "processed", "joined_df.csv")
df_merged.to_csv(joined_df_path)